In [1]:
from ultralytics import YOLO
import os
import torch
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import random
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
def checking_gpu():
    print(torch.cuda.is_available())  # Should be True
    print(torch.cuda.get_device_name(0))  # Prints GPU name
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    torch.cuda.is_available()

    return device

In [3]:
def model_training(model, device):
    results = model.train(
        project="YOLO8m-Experiments",
        name="left_right_signs_train",
        data="config.yaml",
        optimizer="AdamW",
        augment=True,
        fliplr=0.0,
        patience=7,
        epochs=100,  # Number of epochs
        imgsz=800,  # Image size
        batch=8,
        device=device,  # disable left-right flipsoo the left and right images wont get confused
        exist_ok=True,
        degrees=5,  # small tilt, like camera shake
        translate=0.1,  # slight shift
        scale=0.3,
        mosaic=0.0,
        mixup=0.0,
    )

    results = model.val()
    
    return results

In [4]:
def evaluation(results):

    # Print specific metrics
    print("Class indices with average precision:", results.ap_class_index)
    print("Average precision for all classes:", results.box.all_ap)
    print("Average precision:", results.box.ap)
    print("Average precision at IoU=0.50:", results.box.ap50)
    print("Class indices for average precision:", results.box.ap_class_index)
    print("Class-specific results:", results.box.class_result)
    print("F1 score:", results.box.f1)
    print("F1 score curve:", results.box.f1_curve)
    print("Overall fitness score:", results.box.fitness)
    print("Mean average precision:", results.box.map)
    print("Mean average precision at IoU=0.50:", results.box.map50)
    print("Mean average precision at IoU=0.75:", results.box.map75)
    print("Mean average precision for different IoU thresholds:", results.box.maps)
    print("Mean results for different metrics:", results.box.mean_results)
    print("Mean precision:", results.box.mp)
    print("Mean recall:", results.box.mr)
    print("Precision:", results.box.p)
    print("Precision curve:", results.box.p_curve)
    print("Precision values:", results.box.prec_values)
    print("Specific precision metrics:", results.box.px)
    print("Recall:", results.box.r)
    print("Recall curve:", results.box.r_curve)
   

In [5]:
def verify_yolo_labels_random(image_dir, label_dir, class_names, sample_limit=5):
    image_files = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    if len(image_files) == 0:
        print("No images found in the directory.")
        return

    # Select random sample
    random_images = random.sample(image_files, min(sample_limit, len(image_files)))

    for img_file in random_images:
        image_path = os.path.join(image_dir, img_file)
        label_file = os.path.splitext(img_file)[0] + ".txt"
        label_path = os.path.join(label_dir, label_file)

        if not os.path.exists(label_path):
            print(f"No label for {img_file}, skipping.")
            continue

        image = Image.open(image_path).convert("RGB")
        draw = ImageDraw.Draw(image)
        w, h = image.size

        with open(label_path, "r") as f:
            for line in f:
                cls, x, y, bw, bh = map(float, line.strip().split())
                cls = int(cls)
                x1 = (x - bw / 2) * w
                y1 = (y - bh / 2) * h
                x2 = (x + bw / 2) * w
                y2 = (y + bh / 2) * h
                draw.rectangle([x1, y1, x2, y2], outline="red", width=2)
                draw.text((x1, y1), class_names[cls], fill="white")

        plt.figure(figsize=(6, 6))
        plt.title(f"Labeled: {img_file}")
        plt.imshow(image)
        plt.axis("off")
        plt.show()



In [6]:
import torch
torch.cuda.empty_cache()

In [7]:
model = YOLO("yolov8m.pt")  
device=checking_gpu()

results = model_training(model,device)


True
NVIDIA GeForce RTX 3060 Ti
Using device: cuda
New https://pypi.org/project/ultralytics/8.3.151 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.127  Python-3.12.7 torch-2.7.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3060 Ti, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=config.yaml, degrees=5, deterministic=True, device=cuda:0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=0.0, multi_scale=False, name=left_

train: Scanning C:\Users\ai_wo\OneDrive\Desktop\road_signs_detection\Data2\labels\train.cache... 1432 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1432/1432 [00:00<?, ?it/s]


val: Fast image access  (ping: 0.00.0 ms, read: 78.458.9 MB/s, size: 25.7 KB)


val: Scanning C:\Users\ai_wo\OneDrive\Desktop\road_signs_detection\Data2\labels\train.cache... 1432 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1432/1432 [00:00<?, ?it/s]


Plotting labels to YOLO8m-Experiments\left_right_signs_train\labels.jpg... 
optimizer: AdamW(lr=0.01, momentum=0.937) with parameter groups 77 weight(decay=0.0), 84 weight(decay=0.0005), 83 bias(decay=0.0)
Image sizes 800 train, 800 val
Using 8 dataloader workers
Logging results to YOLO8m-Experiments\left_right_signs_train
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...


25 epochs completed in 0.534 hours.
Optimizer stripped from YOLO8m-Experiments\left_right_signs_train\weights\last.pt, 52.0MB
Optimizer stripped from YOLO8m-Experiments\left_right_signs_train\weights\best.pt, 52.0MB

Validating YOLO8m-Experiments\left_right_signs_train\weights\best.pt...
Ultralytics 8.3.127  Python-3.12.7 torch-2.7.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3060 Ti, 8192MiB)
Model summary (fused): 92 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 90/90 [00:32<00:00,  2.75it/s]


                   all       1432       1442      0.932       0.96      0.984      0.815
                  stop        484        494      0.921      0.926      0.978      0.803
           right_right        948        948      0.943      0.994       0.99      0.826
Speed: 0.2ms preprocess, 19.6ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to YOLO8m-Experiments\left_right_signs_train
Ultralytics 8.3.127  Python-3.12.7 torch-2.7.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3060 Ti, 8192MiB)
Model summary (fused): 92 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 315.080.6 MB/s, size: 21.4 KB)


val: Scanning C:\Users\ai_wo\OneDrive\Desktop\road_signs_detection\Data2\labels\train.cache... 1432 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1432/1432 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 179/179 [00:48<00:00,  3.65it/s]


                   all       1432       1442      0.933       0.96      0.984      0.815
                  stop        484        494      0.921      0.926      0.978      0.804
           right_right        948        948      0.944      0.994       0.99      0.826
Speed: 0.2ms preprocess, 31.1ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to YOLO8m-Experiments\left_right_signs_train


In [8]:
evaluation(results)

Class indices with average precision: [0 1]
Average precision for all classes: [[    0.97765     0.97751     0.97569     0.96556     0.96249     0.95023     0.93485     0.83049     0.42203    0.043844]
 [    0.98971     0.98971     0.98971     0.98971     0.98971     0.98776     0.97001     0.87626     0.44589    0.027526]]
Average precision: [    0.80403      0.8256]
Average precision at IoU=0.50: [    0.97765     0.98971]
Class indices for average precision: [0 1]
Class-specific results: <bound method Metric.class_result of ultralytics.utils.metrics.Metric object with attributes:

all_ap: array([[    0.97765,     0.97751,     0.97569,     0.96556,     0.96249,     0.95023,     0.93485,     0.83049,     0.42203,    0.043844],
       [    0.98971,     0.98971,     0.98971,     0.98971,     0.98971,     0.98776,     0.97001,     0.87626,     0.44589,    0.027526]])
ap: array([    0.80403,      0.8256])
ap50: array([    0.97765,     0.98971])
ap_class_index: array([0, 1])
curves: []
curv

In [ ]:
from huggingface_hub import login, upload_file
from dotenv import load_dotenv
import os
# Load .env file
load_dotenv()
# Log in with your HF token
login(token=os.getenv("HUGGINGFACE_TOKEN"))

# Define repo and local file
repo_id = "sue888888888888/yolo_road_signs_detection"
local_model_path = "runs/detect/road_signs_train24/weights/best.pt"

# Upload the model weights
upload_file(
    path_or_fileobj=local_model_path,
    path_in_repo="best.pt",
    repo_id=repo_id,
    repo_type="model"
)


In [ ]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

# Download the file
model_path = hf_hub_download(
    repo_id="sue888888888888/yolo_road_signs_detection",
    filename="best.pt",
    repo_type="model"
)

# Load with YOLO
model = YOLO(model_path)